In [1]:
import os
from pathlib import Path
# Change cwd to the project root (parent of 'notebooks/')
os.chdir(Path.cwd().parent)
Path.cwd()

PosixPath('/Users/jbrandt/code/birddog')

In [2]:
# uncomment to use hosted db
#del os.environ["BIRDDOG_USE_LOCAL_NOCODB"]
os.environ.get("BIRDDOG_USE_LOCAL_NOCODB")

In [7]:
import json
from urllib.parse import urlparse, unquote
import mwparserfromhell

from birddog.runtime import Runtime
from birddog.database import Database
from birddog.database_updater import (
    DatabaseUpdater,
    DatabaseUpdateManager,
    get_child_titles,
    _form_page_info_from_title,
    _normalize_date_string,
    )
from birddog.tracker import PageChangeLog
from birddog.utility import transliterate
from birddog.wiki import (
    WIKI_NAMESPACE,
    API_URL,
    expand_link_target,
    canonicalize_title,
    classify_page,
    page_name,
    )

In [4]:
def clear_db():
    updater = runtime._database_update_manager._updater
    ids=updater._db.get_all_ids("Documents")
    updater._db.delete("Documents", ids)
    ids=updater._db.get_all_ids("Pages")
    updater._db.delete("Pages", ids)
def clear_alerts():
    updater = runtime._database_update_manager._updater
    updater.clear_alerts()

In [ ]:
import random
def deterministic_shuffle(items, seed=42):
    rng = random.Random(seed)   # independent RNG instance
    items = list(items)         # avoid mutating caller’s list
    rng.shuffle(items)
    return items

In [ ]:
runtime = Runtime()
#mgr = DatabaseUpdateManager(Runtime())
#updater = DatabaseUpdater(Runtime())

In [ ]:
with open("var/title_sample.json") as file:
    title_list = json.loads(file.read())
title_list = deterministic_shuffle(sorted(list(set(title_list))))

In [ ]:
#clear_db()

In [ ]:
#clear_alerts()

In [ ]:
runtime.database_update_enabled

In [5]:
#title = "ДАЧкО/8/2/330"
#title = "ДАЧкО/388/1/30"
#title = "ДАОО/359/1/228"
#title = "ДАЧгО/Р-8997/1/120"
#title = "ДАВоО/35/9/318"
title = "ДАЖО/1/74/320"

In [8]:
info = _form_page_info_from_title(title)

2026-02-20 19:36:19,564 [INFO] 
AdaptiveThrottle report:
  host_key                                rps   tokens  blocked_s max_in_flight
  ------------------------------------------------------------------------------
  uk.wikisource.org:api                  4.25     4.00       0.00            4


In [10]:
info

{'title': 'ДАЖО/1/74/320',
 'record': {'description_uk': 'Метричні книги православних церков Овруцького повіту',
  'years': '1857',
  'title': 'ДАЖО/1/74/320',
  'level': 'case',
  'label': 'DAZHO-D/1/74/320',
  'timestamp': '2026-02-20 21:56:05+00:00',
  'availability': 'linked',
  'source_type': 'wiki'},
 'links': {'title': 'Архів:ДАЖО/1/74/320',
  'pageid': 200154,
  'parent': {'title': 'Архів:ДАЖО/1/74', 'exists': True},
  'children': [],
  'category_links': [{'title': 'Архів:Категорія:Роботи_1857-ого',
    'exists': True}],
  'internal_links': [{'title': 'Архів:ДАЖО/1/74/319',
    'exists': True,
    'doc_type': None},
   {'title': 'Архів:ДАЖО/1/74/321', 'exists': True, 'doc_type': None}],
  'commons_links': [{'title': 'c:File:ДАЖО 1-74-0320. 1857. Метричні книги православних церков Овруцького повіту.pdf',
    'url': 'https://commons.wikimedia.org/wiki/File:%D0%94%D0%90%D0%96%D0%9E_1-74-0320._1857._%D0%9C%D0%B5%D1%82%D1%80%D0%B8%D1%87%D0%BD%D1%96_%D0%BA%D0%BD%D0%B8%D0%B3%D0%B8_%D0

In [13]:
info["record"]

{'description_uk': 'Метричні книги православних церков Овруцького повіту',
 'years': '1857',
 'title': 'ДАЖО/1/74/320',
 'level': 'case',
 'label': 'DAZHO-D/1/74/320',
 'timestamp': '2026-02-20 21:56:05+00:00',
 'availability': 'linked',
 'source_type': 'wiki'}

In [14]:
db=Database()

2026-02-20 19:43:16,276 [INFO] 
AdaptiveThrottle report:
  host_key                                rps   tokens  blocked_s max_in_flight
  ------------------------------------------------------------------------------
  nocodb.internal:api                   20.00    39.00       0.00           24
  uk.wikisource.org:api                  4.25     4.00       0.00            4


In [15]:
db.encode_records("Pages",[info["record"]])

[{'description_uk': 'Метричні книги православних церков Овруцького повіту',
  'years': '1857',
  'title': 'ДАЖО/1/74/320',
  'level': 'case',
  'label': 'DAZHO-D/1/74/320',
  'timestamp': '2026-02-20',
  'availability': 'linked',
  'source_type': 'wiki'}]

In [12]:
_normalize_date_string(info["record"]["timestamp"])

'2026-02-20 21:56:05+00:00'

In [ ]:
runtime.update_to_database(title)

In [ ]:
runtime.update_to_database(title_list[100:])

In [ ]:
runtime.update_to_database("ДАОО/1", deep=True)

In [ ]:
runtime.update_to_database('ДАОО/1/230/126')

In [ ]:
runtime._database_update_manager._updater.update_records(['ДАОО/1/191', 'ДАОО/1/229', 'ДАОО/1/124', 'ДАОО/1/166', 'ДАОО/1/230а', 'ДАОО/1/230', 'ДАОО/1/229б', 'ДАОО/1/229а', 'ДАОО/1/228', 'ДАОО/1/2', 'ДАОО/1/152', 'ДАОО/1/154', 'ДАОО/1/150'])

In [ ]:
from birddog.store import get_key_value_store
store = get_key_value_store()
int(store.get("DB Loader", "cursor"))

In [ ]:
#store.remove_all("DB Loader")

In [ ]:
from time import sleep

import json
from birddog.runtime import Runtime
from birddog.database import Database
from birddog.database_updater import DatabaseUpdater
from birddog.tracker import PageChangeLog
from birddog.store import get_key_value_store

updater = DatabaseUpdater(runtime=Runtime())
store = get_key_value_store()
change_log = PageChangeLog()
changes = change_log.get()
update_titles = sorted([title.replace("Архів:", "") for title in changes.keys()])

#def update_batch(updater, titles):
#    updater.update_records(titles)
#    updater.start_translation()


In [ ]:
update_titles = deterministic_shuffle(update_titles)

In [ ]:
len(update_titles)

In [ ]:
# 2026-02-09 09:30:26,617 [ERROR] DatabaseUpdateManager: exception during subtask execution: HTTP 422: {"error":"ERR_INVALID_OFFSET_VALUE","message":"Offset value '50' is invalid"}, {'titles': ['ДАХО/Р-6531/85/125', 'ДАХО/Р-6531/85/130', 'ДАХО/Р-6531/85/131', 'ДАХО/Р-6531/85/132', 'ДАХО/Р-6531/85/133', 'ДАХО/Р-6531/85/134', 'ДАХО/Р-6531/85/135', 'ДАХО/Р-6531/85/136', 'ДАХО/Р-6531/85/137', 'ДАХО/Р-6531/85/138', 'ДАХО/Р-6531/85/139', 'ДАХО/Р-6531/85/140', 'ДАХО/Р-6531/85/141', 'ДАХО/Р-6531/85/142', 'ДАХО/Р-6531/85/143', 'ДАХО/Р-6531/85/144', 'ДАХО/Р-6531/85/145', 'ДАХО/Р-6531/85/146', 'ДАХО/Р-6531/85/147', 'ДАХО/Р-6531/85/148'], 'deep': False}


In [5]:
updater = DatabaseUpdater(runtime=Runtime())

2026-02-10 16:40:46,193 [INFO] PageUpdateManager.init(): detect_environment==local
2026-02-10 16:40:46,693 [INFO] fetch_url: 1 requests in last 60s → 0.02 req/s
2026-02-10 16:40:47,131 [INFO] KillSwitch: loading thresholds from resources/kill_thresholds.json
2026-02-10 16:40:47,131 [INFO] Runtime: truncating log history before 2026-01-27 00:40:47.131851+00:00


In [6]:
titles = ['ДАХО/40/146/165', 'ДАХО/40/146/167', 'ДАХО/40/146/173', 
          'ДАХО/40/146/187', 'ДАХО/40/146/49', 'ДАХО/40/147/12', 
          'ДАХО/40/147/17', 'ДАХО/40/147/27', 'ДАХО/40/147/29', 
          'ДАХО/40/147/30', 'ДАХО/Р-6531/82', 'ДАХО/Р-6531/83', 
          'ДАХО/Р-6531/84', 'ДАХО/Р-6531/85', 'ДАХО/Р-6531/85/211', 
          'ДАХО/Р-6531/85/211а', 'ДАХО/Р-6531/86', 
          'ДАХО/Р-6531/87', 'ДАХО/Р-6531/88', 'ДАХО/Р-6531/89']

In [7]:
updater.update_page_records(titles)

2026-02-10 16:40:53,789 [INFO] update_page_records: 20 titles
2026-02-10 16:40:53,790 [INFO] Updater: accessing wiki page info for 20 pages
2026-02-10 16:40:56,786 [INFO] fetch_url: 17 requests in last 60s → 0.28 req/s
2026-02-10 16:41:04,520 [INFO] Updater: analyzing page links
2026-02-10 16:41:07,047 [INFO] fetch_url: 53 requests in last 60s → 0.88 req/s
2026-02-10 16:41:17,269 [INFO] fetch_url: 81 requests in last 60s → 1.35 req/s
2026-02-10 16:41:27,456 [INFO] fetch_url: 116 requests in last 60s → 1.93 req/s
2026-02-10 16:41:37,545 [INFO] fetch_url: 146 requests in last 60s → 2.43 req/s
2026-02-10 16:41:43,320 [INFO] Updater: linking child pages
2026-02-10 16:41:47,745 [INFO] fetch_url: 172 requests in last 60s → 2.87 req/s
2026-02-10 16:41:49,471 [INFO] Child links for ДАХО/Р-6531/85/211 updated (1 children)
2026-02-10 16:41:57,996 [INFO] fetch_url: 191 requests in last 60s → 3.18 req/s
2026-02-10 16:42:03,595 [INFO] Parent link for ДАХО/Р-6531/85/211а updated (ДАХО/Р-6531/85)
202

True

In [ ]:
titles = [
    'ДАХО/Р-6531/85/125', 'ДАХО/Р-6531/85/130', 'ДАХО/Р-6531/85/131', 'ДАХО/Р-6531/85/132', 'ДАХО/Р-6531/85/133', 
    'ДАХО/Р-6531/85/134', 'ДАХО/Р-6531/85/135', 'ДАХО/Р-6531/85/136', 'ДАХО/Р-6531/85/137', 
    'ДАХО/Р-6531/85/138', 'ДАХО/Р-6531/85/139', 'ДАХО/Р-6531/85/140', 'ДАХО/Р-6531/85/141', 
    'ДАХО/Р-6531/85/142', 'ДАХО/Р-6531/85/143', 'ДАХО/Р-6531/85/144', 'ДАХО/Р-6531/85/145', 
    'ДАХО/Р-6531/85/146', 'ДАХО/Р-6531/85/147', 'ДАХО/Р-6531/85/148']


In [ ]:
updater.update_page_records(titles)

In [ ]:
titles = ['Архів:OMR',
 'Архів:Єврейське_містечко/Подільська_губернія',
 'Архів:Архіви',
 'Архів:ГАДА/354/25',
 'Архів:ДАВіО/172/9',
 'Архів:ДАВіО/22/1',
 'Архів:ДАВіО/24/1',
 'Архів:ДАВіО/904/41',
 'Архів:ДАВіО/904/41/7',
 'Архів:ДАЖО/1/73/500',
 'Архів:ДАЖО/1/73/508',
 'Архів:ДАЖО/1/74/235',
 'Архів:ДАЖО/1/74/236',
 'Архів:ДАЖО/1/74/241',
 'Архів:ДАЖО/1/74/242',
 'Архів:ДАЖО/1/74/301',
 'Архів:ДАЖО/1/74/332',
 'Архів:ДАЖО/1/75/3',
 'Архів:ДАЖО/1/77',
 'Архів:ДАЖО/1/77/1219',
 'Архів:ДАЖО/1/77/1353',
 'Архів:ДАЖО/1/77/1368',
 'Архів:ДАЖО/1/77/1383',
 'Архів:ДАЖО/1/77/1397',
 'Архів:ДАЖО/1/77/1406',
 'Архів:ДАЖО/1/77/1410',
 'Архів:ДАЖО/1/77/1418',
 'Архів:ДАЖО/1/77/1423',
 'Архів:ДАЖО/1/77/1424',
 'Архів:ДАЖО/1/77/1429',
 'Архів:ДАЖО/1/77/1435',
 'Архів:ДАЖО/1/77/1440',
 'Архів:ДАЖО/1/77/1442',
 'Архів:ДАЖО/1/77/1443',
 'Архів:ДАЖО/1/77/1452',
 'Архів:ДАЖО/1/77/1463',
 'Архів:ДАЖО/1/77/1467',
 'Архів:ДАЖО/1/77/1471',
 'Архів:ДАЖО/1/77/1475',
 'Архів:ДАЖО/1/77/1480',
 'Архів:ДАЖО/1/77/1491',
 'Архів:ДАЖО/1/77/1503',
 'Архів:ДАЖО/1/77/1518',
 'Архів:ДАЖО/1/77/1527',
 'Архів:ДАЖО/1/77/1533',
 'Архів:ДАЖО/1/77/1534',
 'Архів:ДАЖО/1/77/1539',
 'Архів:ДАЖО/1/77/1542',
 'Архів:ДАЖО/1/77/1550',
 'Архів:ДАЖО/1/77/1562',
 'Архів:ДАЖО/1/77/1832',
 'Архів:ДАЖО/1/77/1854',
 'Архів:ДАЖО/1/77/1873',
 'Архів:ДАЖО/1/77/1904',
 'Архів:ДАЖО/1/77/1915',
 'Архів:ДАЖО/1/77/1924',
 'Архів:ДАЖО/1/77/1957',
 'Архів:ДАЖО/1/77/2152',
 'Архів:ДАЖО/1/77/2159',
 'Архів:ДАЖО/1/77/434',
 'Архів:ДАЖО/1/78',
 'Архів:ДАЖО/1/78/1101',
 'Архів:ДАЖО/1/78/1106',
 'Архів:ДАЖО/1/81',
 'Архів:ДАЖО/1/81/103',
 'Архів:ДАЖО/1/85',
 'Архів:ДАЖО/1/86',
 'Архів:ДАЖО/1/87/2',
 'Архів:ДАЖО/146/1/1552',
 'Архів:ДАЖО/146/1/1553',
 'Архів:ДАЖО/146/1/1556',
 'Архів:ДАЖО/Р-5069/1/294',
 'Архів:ДАЖО/Р-5069/2',
 'Архів:ДАЖО/Р-5069/2/393',
 'Архів:ДАЖО/Р-5069/2/827',
 'Архів:ДАКО/280/2/471',
 'Архів:ДАКО/384/6/13',
 'Архів:ДАКО/384/8',
 'Архів:ДАКО/782/1',
 'Архів:ДАКО/782/1/13881',
 'Архів:ДАКО/782/1/13882',
 'Архів:ДАКО/782/1/13883',
 'Архів:ДАКО/782/1/13908',
 'Архів:ДАКО/782/1/13909',
 'Архів:ДАКО/782/1/13910',
 'Архів:ДАКО/782/1/13911',
 'Архів:ДАКО/782/1/13912',
 'Архів:ДАКО/782/1/13913',
 'Архів:ДАКО/782/1/13914',
 'Архів:ДАКО/782/1/13915',
 'Архів:ДАКО/782/1/13916',
 'Архів:ДАКО/782/1/13917',
 'Архів:ДАКО/782/1/13918',
 'Архів:ДАКО/782/1/13919',
 'Архів:ДАКО/782/1/2051',
 'Архів:ДАКО/782/1/2052',
 'Архів:ДАКО/782/1/2053',
 'Архів:ДАКО/782/1/2055',
 'Архів:ДАКО/782/1/2056',
 'Архів:ДАКО/782/1/2057']

In [ ]:
updater.update_records(titles)